# Notebook 1: Scene Graphs & spark_dsg

This notebook introduces the core data structure behind SGET: the **3D scene graph**. You'll learn how scene graphs represent environments hierarchically, and how to load, inspect, and manipulate them using the `spark_dsg` Python library.

## Prerequisites
- Virtual environment activated: `source ~/software/mit/virtual-envs/sget/bin/activate`
- SGET and spark_dsg installed: `pip install -e ".[dev]"` and `pip install -e ../Spark-DSG/`

## 1. What is a Scene Graph?

A **3D scene graph** is a hierarchical representation of a 3D environment. Think of it as a structured map that answers questions like "what objects are in this room?" or "which rooms are in this building?"

The graph is organized in **layers**, from most abstract (buildings) to most concrete (objects):

```
Buildings (layer 5)   ── B0
                          │
Rooms (layer 4)       ── R0 ──── R1
                          │        │
Places (layer 3)      ── p0  p1  p2  p3
                          │ \    │ /
Objects (layer 2)     ── O0 O1  O2  O3
```

Each **node** has:
- A 3D position in the world
- Semantic attributes (name, class label, bounding box, etc.)
- Parent-child relationships to nodes in adjacent layers

Each **edge** is either:
- **Intralayer** (siblings): connects nodes in the same layer (e.g., two adjacent rooms)
- **Interlayer** (containment): connects a parent to a child (e.g., a room contains places)

SGET uses the **spark_dsg** library (from MIT SPARK) to represent scene graphs in memory, and serializes them to/from JSON.

## 2. Loading a Scene Graph from JSON

Scene graphs are stored as JSON files. Let's load one from the heracles example data.

In [ ]:
import os

import spark_dsg as dsg

# Load the example scene graph that ships with heracles.
# This was generated by Hydra from a real 3D environment scan.
DSG_PATH = os.path.expanduser(
    "~/software/mit/sget/heracles/heracles/examples/scene_graphs/example_dsg.json"
)

G = dsg.DynamicSceneGraph.load(DSG_PATH)

print(f"Loaded scene graph: {G.num_nodes()} nodes, {G.num_edges()} edges, {G.num_layers()} layers")

## 3. Exploring Layers

A scene graph is organized into **layers**, each representing a different level of spatial abstraction. The layer IDs are numeric — higher means more abstract:

| Layer ID | Name | Description |
|----------|------|-------------|
| 2 | Objects | Individual semantic objects (chairs, trees, etc.) |
| 3 | Places | 3D regions of free space |
| 4 | Rooms | Room-level regions |
| 5 | Buildings | Building-level containers |
| 20 | MeshPlaces | 2D navigable surface segments |

Not every scene graph will have nodes in every layer — it depends on how the graph was generated.

In [ ]:
# Iterate the main (non-partition) layers.
print("Main layers:")
for layer in G.layers:
    print(
        f"  Layer {layer.id} (partition={layer.partition}): "
        f"{layer.num_nodes()} nodes, {layer.num_edges()} intralayer edges"
    )

# MeshPlaces live in a "partition" of layer 3 (partition=1).
# Partitions are sub-layers — spark_dsg uses them for layers that share
# a numeric ID but represent different concepts.
print("\nPartition layers:")
for layer in G.layer_partitions:
    print(f"  Layer {layer.id} (partition={layer.partition}): {layer.num_nodes()} nodes")

## 4. Inspecting Nodes

Every node has a **NodeSymbol** — a compact identifier combining a category character and a numeric index (e.g., `O0` for Object 0, `R1` for Room 1). The category character identifies which layer the node belongs to, but the exact character varies by DSG creator — for example, Objects might use `'O'` or `'o'`, and MeshPlaces might use `'P'` or `'t'`.

Nodes carry **attributes** whose type depends on the layer and the DSG creator:
- All nodes have `position` (3D coordinates), `is_active`, `last_update_time_ns`
- Objects: `ObjectNodeAttributes` (name, color, semantic_label, bounding_box) or `KhronosObjectAttributes` (adds trajectory, timestamps)
- Rooms: `RoomNodeAttributes` (semantic_label, class probabilities)
- MeshPlaces: `Place2dNodeAttributes` (semantic_label, boundary polygon) or `TravNodeAttributes` (radii, traversability states — no semantic_label)
- Places: `PlaceNodeAttributes` (distance to obstacle)

The attribute type name is available via `type(node.attributes).__name__`.

In [ ]:
# Get a single Object node and inspect it.
obj_layer = G.get_layer(dsg.DsgLayers.OBJECTS)
first_obj = next(iter(obj_layer.nodes))

print(f"Node ID:    {first_obj.id}")  # NodeSymbol (e.g., O0)
print(f"  category: {first_obj.id.category}")  # 'O'
print(f"  index:    {first_obj.id.categoryId()}")  # 0
print(f"  uint64:   {first_obj.id.value}")  # Internal numeric encoding

attrs = first_obj.attributes
print(f"\nPosition:       {attrs.position}")  # [x, y, z] numpy array
print(f"Semantic label: {attrs.semantic_label}")  # Integer class ID
print(f"Name:           '{attrs.name}'")

In [ ]:
# Nodes know their parent, children, and siblings.
print(f"Has parent: {first_obj.has_parent()}")
if first_obj.has_parent():
    parent_id = first_obj.get_parent()
    parent_sym = dsg.NodeSymbol(parent_id)
    print(f"Parent:     {parent_sym} (layer {G.get_node(parent_id).layer.layer})")

print(f"Children:   {list(first_obj.children())}")
print(f"Siblings:   {list(first_obj.siblings())}")

In [ ]:
# Let's also look at a Room node — these have different attributes.
room_layer = G.get_layer(dsg.DsgLayers.ROOMS)
first_room = next(iter(room_layer.nodes))

print(f"Room: {first_room.id}")
print(f"  Position:  {first_room.attributes.position}")
print(f"  Label:     {first_room.attributes.semantic_label}")
print(f"  Children:  {len(list(first_room.children()))} nodes")

# Show a few children to see the hierarchy
children = list(first_room.children())[:5]
for cid in children:
    child_sym = dsg.NodeSymbol(cid)
    child_node = G.get_node(cid)
    print(f"    {child_sym} (layer {child_node.layer.layer})")

## 5. Edges — Connectivity and Containment

Scene graph edges come in two flavors:

- **Intralayer edges** (siblings): connect nodes within the same layer. For example, two adjacent rooms share a ROOM_CONNECTED edge. These represent spatial adjacency.
- **Interlayer edges** (containment): connect a parent node to a child node across layers. A room CONTAINS places, a place CONTAINS objects. These encode the hierarchy.

spark_dsg stores intralayer edges on each layer, and interlayer edges on the graph itself.

In [ ]:
# Count intralayer edges per layer.
print("Intralayer edges (sibling connections):")
for layer in G.layers:
    if layer.num_edges() > 0:
        print(f"  Layer {layer.id}: {layer.num_edges()} edges")

# Count interlayer edges.
interlayer_count = sum(1 for _ in G.interlayer_edges)
print(f"\nInterlayer edges (containment): {interlayer_count}")

# Peek at a few interlayer edges.
print("\nSample interlayer edges:")
for i, edge in enumerate(G.interlayer_edges):
    src = dsg.NodeSymbol(edge.source)
    tgt = dsg.NodeSymbol(edge.target)
    src_layer = G.get_node(edge.source).layer.layer
    tgt_layer = G.get_node(edge.target).layer.layer
    print(f"  {src} (layer {src_layer}) → {tgt} (layer {tgt_layer})")
    if i >= 4:
        print(f"  ... and {interlayer_count - 5} more")
        break

## 6. Converting to NetworkX

spark_dsg provides a built-in conversion to NetworkX graphs. This is useful for graph algorithms (shortest path, centrality, etc.) and for computing layout positions — which is exactly how SGET's 2D view works.

The conversion flattens all node attributes into NetworkX node data and includes both intralayer and interlayer edges.

In [ ]:
from spark_dsg.networkx import graph_to_networkx

# Convert the full DSG (including partition layers like MeshPlaces).
nx_graph = graph_to_networkx(G, include_partitions=True)

print(f"NetworkX graph: {nx_graph.number_of_nodes()} nodes, {nx_graph.number_of_edges()} edges")
print(f"Directed: {nx_graph.is_directed()}")  # Always undirected

# Node IDs in the nx graph are uint64 values from NodeSymbol.value.
# To recover the human-readable symbol, wrap in NodeSymbol.
sample_id = list(nx_graph.nodes)[0]
sample_sym = dsg.NodeSymbol(sample_id)
print(f"\nSample node: uint64={sample_id}, symbol={sample_sym}")

# Node data contains all flattened attributes.
sample_data = nx_graph.nodes[sample_id]
print(f"Node data keys: {list(sample_data.keys())}")

## Summary

You've learned:
- A **scene graph** is a hierarchical representation of a 3D environment with layers (Buildings → Rooms → Places → Objects)
- **spark_dsg** loads scene graphs from JSON and provides access to layers, nodes, edges, and attributes
- **NodeSymbol** encodes a category character + index (e.g., `O0`, `R1`)
- Nodes have **parent/child/sibling** relationships that encode the hierarchy
- Edges are either **intralayer** (sibling connectivity) or **interlayer** (containment)
- The graph can be converted to **NetworkX** for layout algorithms

**Next notebook**: [02_neo4j_backend.ipynb](02_neo4j_backend.ipynb) — storing the scene graph in Neo4j and querying it with Cypher.